<a href="https://colab.research.google.com/github/zhadritto/ai-phishing-literature-review/blob/main/notebooks/01_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3 — Reproducible Benchmark Analysis

**Research repository:** `zhadritto/ai-phishing-literature-review`

This notebook runs the repository's technical benchmark in Google Colab, processes the JSONL measurements, excludes warm-up runs from latency statistics, calculates median, percentiles and participant-independent run-level bootstrap confidence intervals, tracks execution errors, and exports the two required figures.

**Important distinction:** this notebook analyzes the performance of the Python data-processing pipeline. It does not analyze student survey accuracy. Survey analysis is a separate research workflow.

Run the notebook from top to bottom using **Runtime → Run all**. The benchmark uses synthetic workload records only; they are not student responses and must not be described as survey findings.

In [1]:
# Cell 1 — Install the one additional configuration dependency
%pip -q install "PyYAML>=6,<7"

In [2]:
# Cell 2 — Set up the repository workspace and download the committed source files
from pathlib import Path
import sys
import urllib.request

REPOSITORY = "zhadritto/ai-phishing-literature-review"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{REPOSITORY}/{BRANCH}/"

WORKSPACE = Path("/content/ai-phishing-literature-review")
FILES_TO_DOWNLOAD = [
    "config/benchmark_config.yaml",
    "src/benchmark.py",
    "src/process_data.py",
    "src/visualize.py",
]

for relative_path in FILES_TO_DOWNLOAD:
    destination = WORKSPACE / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    url = BASE_URL + relative_path
    try:
        with urllib.request.urlopen(url, timeout=30) as response:
            content = response.read()
        destination.write_bytes(content)
        print(f"Downloaded: {relative_path}")
    except Exception as exc:
        raise RuntimeError(
            f"Could not download {relative_path} from GitHub. "
            "Check that the file has been committed to the main branch. "
            f"Original error: {exc}"
        ) from exc

if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

print("\\nWorkspace:", WORKSPACE)
print("Repository files are ready.")

Downloaded: config/benchmark_config.yaml
Downloaded: src/benchmark.py
Downloaded: src/process_data.py
Downloaded: src/visualize.py
\nWorkspace: /content/ai-phishing-literature-review
Repository files are ready.


## 1. Data collection and measurement

The next cell executes the committed `src/benchmark.py`. It creates a synthetic workload at four input sizes and compares two implementations of the same aggregation task (`baseline` and `proposed`). Each method receives five warm-up runs and thirty evaluation runs per input size.

The target function is timed with `time.perf_counter_ns()`. JSONL serialization and file writing happen after the measured interval. Failed runs remain in the logs and are counted by the error-rate guardrail.

In [3]:
# Cell 3 — Run the benchmark and create the raw JSONL log
from src import benchmark

benchmark.main()

RAW_LOG = WORKSPACE / "data/raw/test_run.jsonl"
assert RAW_LOG.exists(), "Benchmark did not create the JSONL file."
print("\\nRaw JSONL file:", RAW_LOG)
print("Raw log size (bytes):", RAW_LOG.stat().st_size)

DATA PROCESSING PERFORMANCE BENCHMARK
Run ID: f90ea985e694
Python: 3.13.18
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
CPU count: 2

Validating implementations...
Baseline and proposed results match.

Processing load size: 100
Warm-up and evaluation completed.

Processing load size: 1000
Warm-up and evaluation completed.

Processing load size: 5000
Warm-up and evaluation completed.

Processing load size: 10000
Warm-up and evaluation completed.

JSONL logs saved to:
/content/ai-phishing-literature-review/data/raw/test_run.jsonl

FINAL BENCHMARK SUMMARY
Total runs: 280
Warm-up runs: 40
Evaluation runs: 240
Successful runs: 280
Failed runs: 0
Execution error rate: 0.00%
All benchmark executions completed successfully.
\nRaw JSONL file: /content/ai-phishing-literature-review/data/raw/test_run.jsonl
Raw log size (bytes): 148985


## 2. Data ingestion and validation

Load the JSONL log and validate the required fields. Warm-up rows are retained in the raw file for transparency, but they will not contribute to latency summaries.

In [4]:
# Cell 4 — Load and validate raw JSONL measurements
from src import process_data

config = process_data.load_config()
raw_data = process_data.load_jsonl(config["raw_file"])
validated_data = process_data.validate_data(raw_data)

print("Total raw records:", len(validated_data))
print("\\nRecords by phase:")
display(validated_data["phase"].value_counts().rename_axis("phase").to_frame("records"))
print("\\nRecords by method and phase:")
display(
    validated_data.groupby(["method", "phase"]).size()
    .rename("records").reset_index()
)

Total raw records: 280
\nRecords by phase:


,records
phase,
benchmark,240
warmup,40


\nRecords by method and phase:


,method,phase,records
0,baseline,benchmark,120
1,baseline,warmup,20
2,proposed,benchmark,120
3,proposed,warmup,20


In [5]:
# Cell 5 — Explicitly separate warm-up and evaluation runs
warmup_data = validated_data[validated_data["phase"] == "warmup"].copy()
evaluation_data = validated_data[validated_data["phase"] == "benchmark"].copy()

assert not evaluation_data.empty, "No evaluation measurements were found."
assert (evaluation_data["phase"] == "benchmark").all()
assert not (evaluation_data["phase"] == "warmup").any()

print("Warm-up records excluded from statistics:", len(warmup_data))
print("Evaluation records used for statistics:", len(evaluation_data))
print("\\nEvaluation runs by method and input size:")
display(
    evaluation_data.groupby(["method", "load_size"])
    .size().rename("evaluation_runs").reset_index()
)

Warm-up records excluded from statistics: 40
Evaluation records used for statistics: 240
\nEvaluation runs by method and input size:


,method,load_size,evaluation_runs
0,baseline,100,30
1,baseline,1000,30
2,baseline,5000,30
3,baseline,10000,30
4,proposed,100,30
5,proposed,1000,30
6,proposed,5000,30
7,proposed,10000,30


## 3. Statistical aggregation

For each method and input size, calculate mean, standard deviation, median, p25, p75, p95 and p99. The 95% confidence intervals are percentile-bootstrap intervals. Failed execution attempts are excluded from latency distribution estimates but remain in the error-rate denominator and in a separate error report.

Because the benchmark compares run-level execution times, the bootstrap resamples successful evaluation runs within each method/load condition. This is distinct from the participant-level bootstrap used for the student survey.

In [6]:
# Cell 6 — Calculate the benchmark summary and guardrail reports
summary = process_data.calculate_summary(validated_data, config)
speedup = process_data.calculate_speedup(validated_data, config)
error_summary = process_data.create_error_summary(validated_data)

print("Latency and error summary:")
display(summary)

print("Paired speedup summary:")
display(speedup)

print("Failed evaluation runs:")
if error_summary.empty:
    print("No failed evaluation runs.")
else:
    display(error_summary)

Latency and error summary:


,method,load_size,evaluation_attempts,expected_evaluation_runs,successful_runs,failed_runs,run_count_complete,success_rate,error_rate,missing_input_rate,...,median_ms,p25_ms,p75_ms,p95_ms,p99_ms,mean_ci_lower_ms,mean_ci_upper_ms,median_ci_lower_ms,median_ci_upper_ms,warmup_runs_excluded
0,baseline,100,30,30,30,0,True,1.0,0.0,0.01,...,2.199244,1.086570,3.069736,6.277883,8.540675,1.968699,3.392798,1.125497,2.986411,5
1,baseline,1000,30,30,30,0,True,1.0,0.0,0.01,...,10.184679,7.502289,14.300228,18.913175,20.416606,9.225280,12.519686,7.892635,12.308680,5
2,baseline,5000,30,30,30,0,True,1.0,0.0,0.01,...,21.710875,14.954550,25.993957,43.086829,60.093841,19.061691,27.397401,19.391933,24.610508,5
3,baseline,10000,30,30,30,0,True,1.0,0.0,0.01,...,20.964243,20.870280,21.243474,24.104281,26.977677,21.094996,22.170419,20.891008,21.123467,5
4,proposed,100,30,30,30,0,True,1.0,0.0,0.01,...,16.464836,10.866354,19.704269,26.903052,40.304089,13.821011,19.312382,13.466290,17.570907,5
5,proposed,1000,30,30,30,0,True,1.0,0.0,0.01,...,32.832842,19.146140,42.045380,53.225035,64.096005,27.221551,37.504281,26.523812,37.981773,5
6,proposed,5000,30,30,30,0,True,1.0,0.0,0.01,...,31.784948,17.515895,40.893416,62.298660,67.833418,27.155963,38.007865,25.980157,37.403539,5
7,proposed,10000,30,30,30,0,True,1.0,0.0,0.01,...,23.239390,22.971959,23.652481,27.337017,35.031281,23.224083,25.136682,23.061729,23.337013,5


Paired speedup summary:


,load_size,paired_successful_runs,median_speedup,ci_lower,ci_upper
0,100,30,0.153995,0.119453,0.178058
1,1000,30,0.357132,0.326699,0.409268
2,5000,30,0.754705,0.665925,0.778077
3,10000,30,0.909579,0.899308,0.920631


Failed evaluation runs:
No failed evaluation runs.


In [7]:
# Cell 7 — Check expected experiment completeness
expected_methods = config["methods"]
expected_loads = config["load_sizes"]
expected_repetitions = config["expected_evaluation_runs"]

expected_groups = len(expected_methods) * len(expected_loads)
assert len(summary) == expected_groups, (
    f"Expected {expected_groups} method/load groups, found {len(summary)}"
)

incomplete = summary[~summary["run_count_complete"]]
if not incomplete.empty:
    print("WARNING: Some method/load groups do not have the expected number of evaluation attempts:")
    display(incomplete[["method", "load_size", "evaluation_attempts", "expected_evaluation_runs"]])
else:
    print(f"All {expected_groups} method/load groups contain {expected_repetitions} evaluation attempts.")

print("\\nOverall failed evaluation attempts:", int((~evaluation_data["success"]).sum()))
print("Overall evaluation error rate (%):", round((~evaluation_data["success"]).mean() * 100, 3))

All 8 method/load groups contain 30 evaluation attempts.
\nOverall failed evaluation attempts: 0
Overall evaluation error rate (%): 0.0


## 4. Baseline versus proposed method

Speedup is defined as `baseline latency / proposed latency`. A value above 1 indicates that the proposed implementation was faster for that paired repetition. Interpret the result together with the error rate and confidence interval; do not report a speedup if the paired successful-run count is insufficient.

In [8]:
# Cell 8 — Present a compact comparison table
comparison_columns = [
    "method", "load_size", "successful_runs", "failed_runs",
    "error_rate", "mean_ms", "std_ms", "median_ms",
    "p25_ms", "p75_ms", "p95_ms", "p99_ms",
    "median_ci_lower_ms", "median_ci_upper_ms"
]

comparison_table = summary[comparison_columns].copy()
comparison_table["error_rate_percent"] = comparison_table["error_rate"] * 100
comparison_table["median_meaning"] = comparison_table.apply(
    lambda row: f"{row['median_ms']:.3f} ms ({row['p25_ms']:.3f}–{row['p75_ms']:.3f})",
    axis=1
)
display(comparison_table)

,method,load_size,successful_runs,failed_runs,error_rate,mean_ms,std_ms,median_ms,p25_ms,p75_ms,p95_ms,p99_ms,median_ci_lower_ms,median_ci_upper_ms,error_rate_percent,median_meaning
0,baseline,100,30,0,0.0,2.642631,2.022197,2.199244,1.086570,3.069736,6.277883,8.540675,1.125497,2.986411,0.0,2.199 ms (1.087–3.070)
1,baseline,1000,30,0,0.0,10.824109,4.714288,10.184679,7.502289,14.300228,18.913175,20.416606,7.892635,12.308680,0.0,10.185 ms (7.502–14.300)
2,baseline,5000,30,0,0.0,22.969908,11.801451,21.710875,14.954550,25.993957,43.086829,60.093841,19.391933,24.610508,0.0,21.711 ms (14.955–25.994)
3,baseline,10000,30,0,0.0,21.550313,1.527426,20.964243,20.870280,21.243474,24.104281,26.977677,20.891008,21.123467,0.0,20.964 ms (20.870–21.243)
4,proposed,100,30,0,0.0,16.404891,7.736258,16.464836,10.866354,19.704269,26.903052,40.304089,13.466290,17.570907,0.0,16.465 ms (10.866–19.704)
5,proposed,1000,30,0,0.0,32.412994,14.646449,32.832842,19.146140,42.045380,53.225035,64.096005,26.523812,37.981773,0.0,32.833 ms (19.146–42.045)
6,proposed,5000,30,0,0.0,32.284909,15.440067,31.784948,17.515895,40.893416,62.298660,67.833418,25.980157,37.403539,0.0,31.785 ms (17.516–40.893)
7,proposed,10000,30,0,0.0,23.998542,2.888974,23.239390,22.971959,23.652481,27.337017,35.031281,23.061729,23.337013,0.0,23.239 ms (22.972–23.652)


## 5. Scientific visualization and export

Both figures are exported as vector PDF and 300-DPI PNG files. Figure 1 shows median processing latency with a 95% bootstrap confidence interval. Figure 2 shows the latency distribution, including the median, interquartile range, outliers, and a mean marker with a 95% bootstrap interval.

In [9]:
# Cell 9 — Generate and save the two required figures
from src import visualize

visualize.main()

FIGURES_DIR = WORKSPACE / "reports/figures"
required_figures = [
    "fig1_latency_vs_load.png",
    "fig1_latency_vs_load.pdf",
    "fig2_distribution_boxplot.png",
    "fig2_distribution_boxplot.pdf",
]

for name in required_figures:
    path = FIGURES_DIR / name
    assert path.exists() and path.stat().st_size > 0, f"Missing or empty figure: {name}"
    print(f"Created: {name} ({path.stat().st_size:,} bytes)")

SCIENTIFIC VISUALIZATION


FileNotFoundError: Metrics summary not found: /content/ai-phishing-literature-review/data/processed/metrics_summary.csv

In [ ]:
# Cell 10 — Display the generated PNG figures
from IPython.display import display, Image

display(Image(filename=str(FIGURES_DIR / "fig1_latency_vs_load.png")))
display(Image(filename=str(FIGURES_DIR / "fig2_distribution_boxplot.png")))

## 6. Interpretation and reporting

Use the generated tables and figures to describe the measured behavior. Do not assume that `proposed` is faster: state what the measurements show. Report latency in milliseconds, input size in records, uncertainty intervals, and error rates. Mention the benchmark environment (Python version, operating system, CPU information) stored in the raw JSONL records.

The benchmark uses synthetic records to compare two local data-processing implementations. These measurements do not describe email-delivery latency, phishing detection effectiveness, or student behavior.

In [ ]:
# Cell 11 — Show environment metadata recorded by the benchmark
metadata_columns = [
    column for column in [
        "python_version", "platform", "machine",
        "processor", "cpu_count", "pandas_version", "numpy_version"
    ]
    if column in validated_data.columns
]

environment = validated_data[metadata_columns].drop_duplicates()
display(environment)

In [ ]:
# Cell 12 — Export all benchmark artifacts into one ZIP for GitHub upload
import zipfile

EXPORT_DIR = Path("/content/task3_benchmark_outputs")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Re-save processed tables to the paths expected by the assignment.
process_data.save_outputs(summary, speedup, error_summary, config)

artifacts = [
    config["raw_file"],
    config["summary_file"],
    config["speedup_file"],
    config["error_file"],
    FIGURES_DIR / "fig1_latency_vs_load.png",
    FIGURES_DIR / "fig1_latency_vs_load.pdf",
    FIGURES_DIR / "fig2_distribution_boxplot.png",
    FIGURES_DIR / "fig2_distribution_boxplot.pdf",
]

zip_path = EXPORT_DIR / "task3_benchmark_outputs.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in artifacts:
        if path.exists():
            archive.write(path, arcname=str(path.relative_to(WORKSPACE)))
        else:
            print("Not included because it does not exist:", path)

print("Export package created:", zip_path)
print("Package size:", f"{zip_path.stat().st_size:,} bytes")

In [ ]:
# Cell 13 — Download the benchmark package to your computer
from google.colab import files
files.download(str(zip_path))

## 7. Final reproducibility checklist

- [ ] Notebook was run from the first cell to the last without errors.
- [ ] Raw JSONL contains both `warmup` and `benchmark` records.
- [ ] Warm-up records are excluded from latency summaries.
- [ ] Mean, SD, median, p25, p75, p95 and p99 are reported.
- [ ] 95% bootstrap confidence intervals are reported.
- [ ] Failed runs are counted in the error-rate guardrail.
- [ ] Baseline and proposed outputs are validated as equivalent.
- [ ] Both figures exist in PDF and 300-DPI PNG formats.
- [ ] Raw logs and aggregate outputs are clearly identified as synthetic pipeline benchmark data, not student survey results.

**Submission note:** Save this executed notebook to the repository as `notebooks/01_data_analysis.ipynb`. Upload the exported benchmark artifacts to their matching repository folders. Keep the original Google Forms export and participant-level survey data private.